In [1]:
import json
import pandas as pd
import numpy as np

In [2]:
def load_arxiv(filepath, max_rows = 10000):
    records = []
    with open(filepath, "r") as f :
        for i, line in enumerate(f):
            if max_rows and i>=max_rows:
                break
            records.append(json.loads(line))
    return records

records = load_arxiv("/kaggle/input/datasets/organizations/Cornell-University/arxiv/arxiv-metadata-oai-snapshot.json", max_rows=10000)
print(f"loaded {len(records)} records")

loaded 10000 records


In [3]:
def inspect_schema(records, sample_n=3):
    sample = records[0]
    print("="*60)
    print("TOP_LEVEL KEYS & TYPES")
    print("="*60)
    for key, value in sample.items():
        print(f" {key:<20}{type(value).__name__:<15} => {str(value)[:80]}")
    print("\n"+"="*60)
    print(f"SAMPLE RECORDS (n = {sample_n}")
    print("="*60)
    for i,rec in enumerate(records[:sample_n]):
        print(f"\n--- Record {i+1} ---")
        for key, value in rec.items():
            print(f" {key}:{str(value)[:120]}")

inspect_schema(records)

TOP_LEVEL KEYS & TYPES
 id                  str             => 0704.0001
 submitter           str             => Pavel Nadolsky
 authors             str             => C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-P. Yuan
 title               str             => Calculation of prompt diphoton production cross sections at Tevatron and
  LHC e
 comments            str             => 37 pages, 15 figures; published version
 journal-ref         str             => Phys.Rev.D76:013009,2007
 doi                 str             => 10.1103/PhysRevD.76.013009
 report-no           str             => ANL-HEP-PR-07-12
 categories          str             => hep-ph
 license             NoneType        => None
 abstract            str             =>   A fully differential calculation in perturbative quantum chromodynamics is
pre
 versions            list            => [{'version': 'v1', 'created': 'Mon, 2 Apr 2007 19:18:42 GMT'}, {'version': 'v2',
 update_date         str             => 2008-11-26
 au

In [4]:
df = pd.DataFrame(records)
print("\nDataFrame shape:", df.shape)
print("\nColumn dtypes:\n", df.dtypes)


DataFrame shape: (10000, 14)

Column dtypes:
 id                object
submitter         object
authors           object
title             object
comments          object
journal-ref       object
doi               object
report-no         object
categories        object
license           object
abstract          object
versions          object
update_date       object
authors_parsed    object
dtype: object


In [5]:
def analyze_schema(df):
    print("\n"+"="*60)
    print("Schema Analysis")
    print("="*60)
    
    for col in df.columns:
        null_count  = df[col].isna().sum()
        null_pct    = round(null_count/len(df)*100,2)
        try:
            unique  = df[col].nunique()
        except TypeError:
            unique  = df[col].apply(lambda x: str(x)).nunique()
        sample_vals = df[col].dropna().head(2).tolist()
        
        print(f"\n Column    : {col}")
        print(f" Dtype       : {df[col].dtype}")
        print(f" Nulls       : {null_count}({null_pct}%)")
        print(f" Unique Vals : {unique}")
        print(f" Samples     : {str(sample_vals)[:120]}")

analyze_schema(df)


Schema Analysis

 Column    : id
 Dtype       : object
 Nulls       : 0(0.0%)
 Unique Vals : 10000
 Samples     : ['0704.0001', '0704.0002']

 Column    : submitter
 Dtype       : object
 Nulls       : 0(0.0%)
 Unique Vals : 8031
 Samples     : ['Pavel Nadolsky', 'Louis Theran']

 Column    : authors
 Dtype       : object
 Nulls       : 0(0.0%)
 Unique Vals : 9417
 Samples     : ["C. Bal\\'azs, E. L. Berger, P. M. Nadolsky, C.-P. Yuan", 'Ileana Streinu and Louis Theran']

 Column    : title
 Dtype       : object
 Nulls       : 0(0.0%)
 Unique Vals : 9998
 Samples     : ['Calculation of prompt diphoton production cross sections at Tevatron and\n  LHC energies', 'Sparsity-certifying Graph 

 Column    : comments
 Dtype       : object
 Nulls       : 1138(11.38%)
 Unique Vals : 6799
 Samples     : ['37 pages, 15 figures; published version', 'To appear in Graphs and Combinatorics']

 Column    : journal-ref
 Dtype       : object
 Nulls       : 4566(45.66%)
 Unique Vals : 5430
 Samples     

In [6]:
def parse_authors(raw):
    if not raw:
        return []
    if isinstance(raw,list):
        authors=[]
        for entry in raw:
            if isinstance(entry,list):
                name="".join(part for part in entry if part).strip()
            else: 
                name = str(entry).strip()
            if name:
                authors.append(name)
        return authors
    if isinstance(raw,str) and raw.startswith("["):
        try:
            parsed = ast.literal_eval(raw)
            return parsed_authors(parsed)
        except Exception:
            pass
    if isinstance(raw,str):
        return [a.strip() for a in raw.split(",") if a.strip()]
    return []

def parse_first_date(versions):
    try:
        return versions[0].get("created",None) if versions else None
    except: 
        return None

def parse_categories(cat_str):
    try:
        return cat_str.strip().split()
    except:
        return []

df["authors_parsed"]    = df["authors"].apply(parse_authors)
df["first_submitted"]   = df["versions"].apply(parse_first_date)
df["categories_parsed"] = df["categories"].apply(parse_categories)
df["first_submitted"]   = pd.to_datetime(df["first_submitted"], errors = "coerce")

print("\nParsed fields sample:")
print(df[["id","authors_parsed","first_submitted","categories_parsed"]].head(3))


Parsed fields sample:
          id                                     authors_parsed  \
0  0704.0001  [C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C....   
1  0704.0002                  [Ileana Streinu and Louis Theran]   
2  0704.0003                                      [Hongjun Pan]   

      first_submitted categories_parsed  
0 2007-04-02 19:18:42          [hep-ph]  
1 2007-03-31 02:26:18  [math.CO, cs.CG]  
2 2007-04-01 20:46:54  [physics.gen-ph]  


In [7]:
#filter - to be decided
#for testing purpose using onlu AI/ML categories

ML_CATEGORIES = {"cs.LG","cs.AI","stat.ML","cs.CL","cs.CV","cs.NE"}

def is_ml_paper(cats):
    return bool(set(cats) & ML_CATEGORIES)

df_ml = df[df["categories_parsed"].apply(is_ml_paper).copy()]
print(f"\nAI/ML papers: {len(df_ml)}/{len(df)}total")


AI/ML papers: 110/10000total


In [8]:
def print_sample_row(df,index=0):
    row = df.iloc[index]
    print("="*60)
    print("\nSample row:")
    print("="*60)
    
    print(f"paper_id    : {row['paper_id']}")
    print(f"title       : {row['title']}")
    print(f"date        : {row['date']}")
    print(f"journal_ref : {row['journal_ref'] if pd.notna(row['journal_ref']) else 'N/A'}")
    print(f"doi         : {row['doi'] if pd.notna(row['doi']) else 'N/A'}")
    
    abstract = row['abstract'].replace("\n","").strip()
    print(f"\nabstract   :")
    for i in range(0, min(len(abstract),240),80):
        print(f" {abstract[i:i+80]}")
    if len(abstract)>240:
        print(f"  ...({len(abstract)} chars total")
    
    print(f"\nauthors     : ({len(row['authors'])} total)")
    for a in row['authors'][:5]:
        print(f"  -{a}")
    if len(row['authors'])>5:
        print(f"  ... and {len(row['authors'])-5} more")
    
    print(f"\ncategories  : {','.join(row['categories'])}")
    print("="*60)

In [9]:
KEEP_COLS = {
    "id"               : "Unique ArXiv papers ID",
    "title"            : "Paper Title",
    "abstract"         : "Full abstract text (main embeddings input)",
    "authors_parsed"   : "List of Author Names",
    "categories_parsed" : "List of ArXiv category codes",
    "first_submitted"  : "First Submission date",
    "journal-ref"      : "Journal reference if published",
    "doi"              : "DOI Link"
}

df_clean = df_ml[list(KEEP_COLS.keys()).copy()]
df_clean.columns = [
    "paper_id","title","abstract",
    "authors","categories",
    "date","journal_ref","doi"
]

print("\n"+"="*60)
print("Final Clean Schema")
print("="*60)
for new_col, desc in zip(df_clean.columns, KEEP_COLS.values()):
    print(f"{new_col:<20} -- {desc}")

print_sample_row(df_clean)


Final Clean Schema
paper_id             -- Unique ArXiv papers ID
title                -- Paper Title
abstract             -- Full abstract text (main embeddings input)
authors              -- List of Author Names
categories           -- List of ArXiv category codes
date                 -- First Submission date
journal_ref          -- Journal reference if published
doi                  -- DOI Link

Sample row:
paper_id    : 0704.0047
title       : Intelligent location of simultaneously active acoustic emission sources:
  Part I
date        : 2007-04-01 13:06:50
journal_ref : N/A
doi         : N/A

abstract   :
 The intelligent acoustic emission locator is described in Part I, while PartII d
 iscusses blind source separation, time delay estimation and location of twosimul
 taneously active continuous acoustic emission sources.  The location of acoustic
  ...(1157 chars total

authors     : (1 total)
  -T. Kosel and I. Grabec

categories  : cs.NE,cs.AI


In [10]:
df_clean.to_parquet("arxiv_ml_clean.parquet", index = False)
print("\nSaved to arxiv_ml_clean.parquet")
print(f"Final Shape:{df_clean.shape}")


Saved to arxiv_ml_clean.parquet
Final Shape:(110, 8)
